# testing

In [4]:
# install the necessary libraries

!pip install requests beautifulsoup4 pandas

In [5]:
# import the necessary libraries into the Jupyter notebook
    # requests: downloads the HTML of a page
    # BeautifulSoup: parses and navigates the HTML structure
    # pandas: used to analyse the data

import requests
from bs4 import BeautifulSoup
import pandas as pd

In [6]:
# This is the URL of the webpage I'll be scraping for data

url = "https://worldathletics.org/stats-zone" 

In [7]:
# Send a GET request to the WorldAthletics website; this is grabbing data from the website
# returns a response object which contains the HTML content of the page as a string (response.text) and the HTTP status code (tells me whether the request worked or not)

response = requests.get(url)

In [8]:
if response.status_code == 200:
    print("Request successful")
else:
    print("Request failed")

Request successful


In [9]:
# Covert the raw HTML into a structured object using BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

In [10]:
# Extracting the data

rows = soup.find_all("pre")[0].text.split("\n")

IndexError: list index out of range

In [11]:
print(soup.find_all("pre"))

[]


In [12]:
# Unsuccessful because of the nature of the World Athletics website.

In [13]:
# Assigning the URL of the webpage I'll be scraping to a variable

url_100W = "https://www.alltime-athletics.com/w_100ok.htm"

In [14]:
# Send a HTTP GET request to Alltime Athletics

response_100W = requests.get(url_100W)

In [15]:
# Converting the raw HTML to a structured, searchable object

soup_100W = BeautifulSoup(response_100W.text, "html.parser")

In [16]:
# Creating usable data rows

pre_tag_100W = soup_100W.find("pre")

rows_100W = pre_tag_100W.text.split("\n")

In [17]:
# How many rows are there and what do they look like?

print(len(rows_100W))
print(rows_100W[:5])

3411
['', '        1      10.54      +0.9    Elaine Thompson-Herah          JAM     28.06.92    1      Eugene                     21.08.2021', '        2      10.60      +1.7    Shelly-Ann Fraser-Pryce        JAM     27.12.86    1rA    Lausanne                   26.08.2021', '        3      10.61      +1.2    Florence Griffith-Joyner       USA     21.12.59    1      Indianapolis               17.07.1988', '        3      10.61      -0.6    Elaine Thompson-Herah          JAM     28.06.92    1      Tokyo                      31.07.2021']


In [18]:
# Print the first 5 rows raw so I can see exactly what I'm working with

for row in rows_100W[:5]:
    print(repr(row))
    print(row.split())
    print()

''
[]

'        1      10.54      +0.9    Elaine Thompson-Herah          JAM     28.06.92    1      Eugene                     21.08.2021'
['1', '10.54', '+0.9', 'Elaine', 'Thompson-Herah', 'JAM', '28.06.92', '1', 'Eugene', '21.08.2021']

'        2      10.60      +1.7    Shelly-Ann Fraser-Pryce        JAM     27.12.86    1rA    Lausanne                   26.08.2021'
['2', '10.60', '+1.7', 'Shelly-Ann', 'Fraser-Pryce', 'JAM', '27.12.86', '1rA', 'Lausanne', '26.08.2021']

'        3      10.61      +1.2    Florence Griffith-Joyner       USA     21.12.59    1      Indianapolis               17.07.1988'
['3', '10.61', '+1.2', 'Florence', 'Griffith-Joyner', 'USA', '21.12.59', '1', 'Indianapolis', '17.07.1988']

'        3      10.61      -0.6    Elaine Thompson-Herah          JAM     28.06.92    1      Tokyo                      31.07.2021'
['3', '10.61', '-0.6', 'Elaine', 'Thompson-Herah', 'JAM', '28.06.92', '1', 'Tokyo', '31.07.2021']



In [19]:
# Converting rows of data into a DataFrame

data_100W = []

for row in rows_100W:
    # Split the row by spaces
    parts = row.split()
    # Skip rows without enough elements (I have 9 variables) or header rows
    if len(parts) < 9 or not parts[0].isdigit():
        continue
    try:
        # the columns I know the index of for certain
        rank = parts[0]
        time = parts[1]
        year = parts[-1]
        # Athlete is everything between columns 3 and 5
        athlete = " ".join(parts[3:-5])
        
        data_100W.append([rank, time, athlete, year])
    # if anything fails, skip the row and carry on
    except:
        continue

df_100W = pd.DataFrame(data_100W, columns = ["Rank", "Time", "Athlete", "Year"])

In [20]:
# Checking the structure of the DataFrame

print(df_100W.info())

<class 'pandas.DataFrame'>
RangeIndex: 3408 entries, 0 to 3407
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Rank     3408 non-null   str  
 1   Time     3408 non-null   str  
 2   Athlete  3408 non-null   str  
 3   Year     3408 non-null   str  
dtypes: str(4)
memory usage: 106.6 KB
None


In [21]:
# checking there are no nested lists

print(data_100W[:5])

[['1', '10.54', 'Elaine Thompson-Herah', '21.08.2021'], ['2', '10.60', 'Shelly-Ann Fraser-Pryce', '26.08.2021'], ['3', '10.61', 'Florence Griffith-Joyner', '17.07.1988'], ['3', '10.61', 'Elaine Thompson-Herah', '31.07.2021'], ['3', '10.61', 'Melissa Jefferson-Wooden', '14.09.2025']]


In [22]:
df_100W.head()

,Rank,Time,Athlete,Year
0,1,10.54,Elaine Thompson-Herah,21.08.2021
1,2,10.60,Shelly-Ann Fraser-Pryce,26.08.2021
2,3,10.61,Florence Griffith-Joyner,17.07.1988
3,3,10.61,Elaine Thompson-Herah,31.07.2021
4,3,10.61,Melissa Jefferson-Wooden,14.09.2025


In [23]:
# Checking for lost data: How many rows did I lose?

#Count the number of valid rows on AllTimeAthletics
valid_rows_100W = []

for row in rows_100W:
    parts = row.split()
    if len(parts) >= 9 and parts[0].isdigit():
        valid_rows_100W.append(row)

print("Estimated valid rows:", len(valid_rows_100W))

# Compare rows before and after parsing:
print("Valid rows before parsing:", len(valid_rows_100W))
print("Successfully parsed rows:", len(data_100W))

Estimated valid rows: 3408
Valid rows before parsing: 3408
Successfully parsed rows: 3408


In [24]:
# Convert data types within the DataFrame

# Some time values have letters as well (either from wind readings or rankings). 
# I need to clean the strings before converting to float
df_100W["Time"] = df_100W["Time"].str.extract(r"(\d+\.\d+)")
# Then convert to float
df_100W["Time"] = df_100W["Time"].astype(float)

# Values in the year colume are formatted as (e.g.) 21.08.2021.
# I need to clean the dates to only include the year
df_100W["Year"] = df_100W["Year"].str.extract(r"(\d{4})$")
# Then convert to integer
df_100W["Year"] = df_100W["Year"].astype(int)

In [25]:
# Check this was successful

print(df_100W.dtypes)

Rank           str
Time       float64
Athlete        str
Year         int64
dtype: object


In [26]:
df_100W.head()

,Rank,Time,Athlete,Year
0,1,10.54,Elaine Thompson-Herah,2021
1,2,10.60,Shelly-Ann Fraser-Pryce,2021
2,3,10.61,Florence Griffith-Joyner,1988
3,3,10.61,Elaine Thompson-Herah,2021
4,3,10.61,Melissa Jefferson-Wooden,2025
